In [1]:
import pandas as pd 
import numpy as np 
df = pd.read_csv(    "../data/cleaned/paimana_cleaned1.csv" ) 
print("Dataset loaded successfully!") 
print("Rows:", len(df)) 
print("Columns:", len(df.columns))

Dataset loaded successfully!
Rows: 7594
Columns: 20


In [3]:
# CONVERT DATE COLUMNS 
date_columns = [    "DATE_OF_APPROVAL",    "START_DATE",    "ORIGINAL_TARGET_DOC",    "REVISED_DOC" ] 
for col in date_columns:    
    if col in df.columns:        
        df[col] = pd.to_datetime(            
            df[col],            
            errors="coerce"        
    ) 
print("Date conversion completed.")

Date conversion completed.


In [4]:
# STANDARDIZE REPORT MONTH 
df["REPORT_MONTH"] = (
     df["REPORT_MONTH"]    
     .astype(str)    
     .str.strip()    
     .str.upper() ) 
month_order = [    
    "APRIL",    
    "MAY",    
    "JUNE",    
    "JULY" ] 
df["REPORT_MONTH"] = pd.Categorical(    
    df["REPORT_MONTH"],    
    categories=month_order,    
    ordered=True )

In [5]:
# SORT PROJECT HISTORY 
df = df.sort_values(    ["PROJECT_ID", "REPORT_MONTH"] ).reset_index(drop=True)

In [6]:
# Feature 1 — Cost Overrun %
# COST OVERRUN % 
df["COST_OVERRUN_PCT"] = np.where(    
    df["ORIGINAL_COST_RS_CRORE"] > 0,    
    (
        (df["REVISED_COST_RS_CRORE"] - df["ORIGINAL_COST_RS_CRORE"]) 
        / 
        df["ORIGINAL_COST_RS_CRORE"] 
    ) * 100,
     np.nan ) 
print("COST_OVERRUN_PCT created.")

COST_OVERRUN_PCT created.


In [7]:
# Feature 2 — Expenditure %
# EXPENDITURE % 
df["EXPENDITURE_PCT"] = np.where(    
    df["ORIGINAL_COST_RS_CRORE"] > 0,    
    (        
        df["CUMULATIVE_EXPENDITURE_RS_CRORE"]        
        /        
        df["ORIGINAL_COST_RS_CRORE"]    
    ) * 100,    
        np.nan 
) 
print("EXPENDITURE_PCT created.")

EXPENDITURE_PCT created.


In [8]:
# Feature 3 — Progress gap
# PROGRESS GAP 
df["PROGRESS_GAP"] = (    
    df["EXPENDITURE_PCT"]
    -     
    df["PHYSICAL_PROGRESS_PERCENT"] ) 
print("PROGRESS_GAP created.")

PROGRESS_GAP created.


In [27]:
# Feature 4 — Project duration
df["START_DATE"] = pd.to_datetime(df["START_DATE"], format="mixed")
df["ORIGINAL_TARGET_DOC"] = pd.to_datetime(df["ORIGINAL_TARGET_DOC"], format="mixed")
df["ORIGINAL_DURATION_DAYS"] = (df["ORIGINAL_TARGET_DOC"] - df["START_DATE"]).dt.days

In [9]:
# Feature 5 — Schedule change
# SCHEDULE CHANGE 
df["SCHEDULE_CHANGE_DAYS"] = (    
    df["REVISED_DOC"]   
    -     
    df["ORIGINAL_TARGET_DOC"] ).dt.days 
print("SCHEDULE_CHANGE_DAYS created.")

SCHEDULE_CHANGE_DAYS created.


In [10]:
# Feature 6 — Progress change
# MONTHLY PHYSICAL PROGRESS CHANGE \
df["PROGRESS_CHANGE"] = (    
    df.groupby("PROJECT_ID")[        
        "PHYSICAL_PROGRESS_PERCENT"    
    ].diff() ) 
print("PROGRESS_CHANGE created.")

PROGRESS_CHANGE created.


In [11]:
# Feature 7 — Expenditure change
# MONTHLY EXPENDITURE CHANGE 
df["EXPENDITURE_CHANGE"] = (    
    df.groupby("PROJECT_ID")[        
        "CUMULATIVE_EXPENDITURE_RS_CRORE"    
    ].diff() ) 
print("EXPENDITURE_CHANGE created.")

EXPENDITURE_CHANGE created.


In [12]:
# Feature 8 — Revised cost change
# MONTHLY REVISED COST CHANGE 
df["REVISED_COST_CHANGE"] = (    
    df.groupby("PROJECT_ID")[        
        "REVISED_COST_RS_CRORE"    ].diff() ) 
print("REVISED_COST_CHANGE created.")

REVISED_COST_CHANGE created.


In [13]:
# Feature 9 — Delay indicator
# DELAY INDICATOR 
df["DELAY_INDICATOR"] = np.where(    
    df["SCHEDULE_CHANGE_DAYS"] > 0,    
    1,    
    0 ) 
print("DELAY_INDICATOR created.")

DELAY_INDICATOR created.


In [16]:
# ========================================= 
# ML PREDICTOR DATASET 
# ========================================== 
predictor_columns = [    
    "PROJECT_ID",    
    "MINISTRY",    
    "SECTOR",
    "STATE",    
    "AGENCY",    
    "ORIGINAL_COST_RS_CRORE",    
    "CUMULATIVE_EXPENDITURE_RS_CRORE",    
    "PHYSICAL_PROGRESS_PERCENT",    
    "START_DATE",    
    "ORIGINAL_TARGET_DOC",    
    "PROGRESS_CHANGE",    
    "EXPENDITURE_CHANGE" 
    ] 
predictor_columns = [    
    col for col in predictor_columns    
    if col in df.columns 
    ] 
ml_features = df[    
    predictor_columns 
    ].copy() 
print("ML predictor dataset created.") 
print("Shape:", ml_features.shape) 
print("Columns:") 
print(ml_features.columns.tolist())

ML predictor dataset created.
Shape: (7594, 12)
Columns:
['PROJECT_ID', 'MINISTRY', 'SECTOR', 'STATE', 'AGENCY', 'ORIGINAL_COST_RS_CRORE', 'CUMULATIVE_EXPENDITURE_RS_CRORE', 'PHYSICAL_PROGRESS_PERCENT', 'START_DATE', 'ORIGINAL_TARGET_DOC', 'PROGRESS_CHANGE', 'EXPENDITURE_CHANGE']


In [17]:
# SAVE ML PREDICTOR DATASET 
ml_features.to_csv("../data/processed/ml_predictor_features.csv",    index=False ) 
print(    "ml_predictor_features.csv saved successfully." )

ml_predictor_features.csv saved successfully.


In [18]:
# ========================================== 
# TARGET DATASET 
# ========================================== 
target_columns = [    
    "PROJECT_ID",    
    "REPORT_MONTH",    
    "COST_OVERRUN_PCT",    
    "SCHEDULE_CHANGE_DAYS",    
    "DELAY_INDICATOR" ] 
target_columns = [    
    col for col in target_columns    
    if col in df.columns ] 
targets = df[    
    target_columns ].copy() 
print("Target dataset created.") 
print(targets.head())

Target dataset created.
   PROJECT_ID REPORT_MONTH  COST_OVERRUN_PCT  SCHEDULE_CHANGE_DAYS  \
0    108841.0         JUNE               0.0                   NaN   
1    108841.0         JULY               0.0                   NaN   
2    124199.0         JULY               0.0                   NaN   
3    171218.0         JULY               0.0                   NaN   
4    175860.0         JULY               0.0                   NaN   

   DELAY_INDICATOR  
0                0  
1                0  
2                0  
3                0  
4                0  


In [19]:
targets.to_csv(    "../data/processed/project_targets.csv",    index=False )
print("project_targets.csv saved successfully.")

project_targets.csv saved successfully.


In [20]:
# ========================================== 
# FINAL FEATURE CHECK 
# ========================================== 
feature_columns = [    
    "COST_OVERRUN_PCT",    
    "EXPENDITURE_PCT",
    "PROGRESS_GAP",    
    "SCHEDULE_CHANGE_DAYS",    
    "PROGRESS_CHANGE",    
    "EXPENDITURE_CHANGE",    
    "REVISED_COST_CHANGE",    
    "DELAY_INDICATOR" ] 
print("FEATURE CHECK") 
print("==============================") 

for col in feature_columns:    
    if col in df.columns:        
        print(            
            f"{col}:",            
            "OK"        
        )    
    else:        
        print(            
            f"{col}: MISSING"        
        )

FEATURE CHECK
COST_OVERRUN_PCT: OK
EXPENDITURE_PCT: OK
PROGRESS_GAP: OK
SCHEDULE_CHANGE_DAYS: OK
PROGRESS_CHANGE: OK
EXPENDITURE_CHANGE: OK
REVISED_COST_CHANGE: OK
DELAY_INDICATOR: OK


In [25]:
print("==========================================") 
print("MEMBER 1 FINAL VALIDATION") 
print("==========================================")


print("Total rows:", len(df)) 
print("Unique projects:", df["PROJECT_ID"].nunique()) 

print("\nMONTHS:") 
print(df["REPORT_MONTH"].value_counts()) 

print("\nREQUIRED FEATURES:") 

required_features = [    
    "COST_OVERRUN_PCT",    
    "EXPENDITURE_PCT",    
    "PROGRESS_GAP",    
    "SCHEDULE_CHANGE_DAYS",    
    "PROGRESS_CHANGE",    
    "EXPENDITURE_CHANGE",    
    "REVISED_COST_CHANGE",    
    "DELAY_INDICATOR" ] 
for feature in required_features:    
    print(        
        feature,        
        "->",        
        "OK" 
        if feature in df.columns 
        else "MISSING"    ) 

print("\nML predictor dataset:") 
print(ml_features.shape) 
print("\nTarget dataset:") 
print(targets.shape) 
print("\n==========================================") 
print("MEMBER 1 VALIDATION COMPLETE") 
print("==========================================")

MEMBER 1 FINAL VALIDATION
Total rows: 7594
Unique projects: 2073

MONTHS:
REPORT_MONTH
MAY      1987
APRIL    1981
JUNE     1847
JULY     1775
Name: count, dtype: int64

REQUIRED FEATURES:
COST_OVERRUN_PCT -> OK
EXPENDITURE_PCT -> OK
PROGRESS_GAP -> OK
SCHEDULE_CHANGE_DAYS -> OK
PROGRESS_CHANGE -> OK
EXPENDITURE_CHANGE -> OK
REVISED_COST_CHANGE -> OK
DELAY_INDICATOR -> OK

ML predictor dataset:
(7594, 12)

Target dataset:
(7594, 5)

MEMBER 1 VALIDATION COMPLETE
